# 10 하이브리드 수요예측 프레임워크

**Phase 1** (단일 모델 baseline) vs **Phase 2** (클러스터별 최적 모델 하이브리드), **SBC vs ML** 클러스터링을 type별로 비교합니다.

- 주요 지표: **WMAPE**
- 클러스터: SBC (`sbc_cluster_type_family.parquet`) vs ML (`ml_cluster_type_family.parquet`)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED, SBC_CLUSTER, ML_CLUSTER
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.metrics import wmape
from utils.forecasting import forecast_arima, forecast_prophet, FORECASTERS

df = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')
sbc = pd.read_parquet(SBC_CLUSTER)
ml = pd.read_parquet(ML_CLUSTER)

df = df.merge(sbc[['type','family','SBC_CLUSTER']], on=['type','family'], how='left')
df = df.merge(ml[['type','family','ML_CLUSTER']], on=['type','family'], how='left')
HORIZON = len(VAL_WEEKS)

# Phase1 후보 모델 (07 결과가 있으면 로드)
ml_sum_path = DATA_PROCESSED / 'forecast_ml_summary.csv'
if ml_sum_path.exists():
    ml_sum = pd.read_csv(ml_sum_path)
    phase1_models = ['arima', 'prophet', 'sba', 'tsb']
    best_phase1 = ml_sum[ml_sum['model'].str.lower().isin(phase1_models)].sort_values('wmape_mean').iloc[0]['model'].lower()
else:
    best_phase1 = 'sba'
print('Phase1 baseline model:', best_phase1)

In [ ]:
# 클러스터별 모델 매핑 (간헐/루미 → SBA·TSB, Smooth → ARIMA/Prophet)
CLUSTER_MODEL = {
    1: 'arima',   # Smooth
    2: 'sba',     # Intermittent
    3: 'prophet', # Erratic
    4: 'tsb',     # Lumpy
}

def forecast_one(series, weeks, model_name, horizon):
    if model_name == 'prophet':
        return forecast_prophet(weeks, series, horizon)
    return FORECASTERS[model_name](series, horizon)

def run_hybrid(cluster_col, label):
    rows = []
    for (typ, fam), g in tqdm(df.groupby(['type','family']), desc=label):
        g_train = g[g['yearweek'] <= TRAIN_WEEK_MAX]
        g_val = g[g['yearweek'].isin(VAL_WEEKS)].sort_values('yearweek')
        if len(g_val) == 0:
            continue
        y_true = g_val['sales'].values
        cluster = int(g[cluster_col].iloc[0])
        model = CLUSTER_MODEL.get(cluster, best_phase1)

        pred = forecast_one(g_train['sales'], g_train['yearweek'], model, HORIZON)
        if len(pred) != len(y_true):
            pred = np.resize(pred, len(y_true))
        rows.append({
            'framework': label, 'type': typ, 'family': fam,
            'cluster': cluster, 'model': model,
            'wmape': wmape(y_true, pred),
        })
    return pd.DataFrame(rows)

phase2_sbc = run_hybrid('SBC_CLUSTER', 'Phase2_SBC')
phase2_ml = run_hybrid('ML_CLUSTER', 'Phase2_ML')
phase2 = pd.concat([phase2_sbc, phase2_ml], ignore_index=True)
phase2.groupby('framework')['wmape'].mean()

In [ ]:
# Phase1: 단일 baseline
rows_p1 = []
for (typ, fam), g in df.groupby(['type','family']):
    g_train = g[g['yearweek'] <= TRAIN_WEEK_MAX]
    g_val = g[g['yearweek'].isin(VAL_WEEKS)].sort_values('yearweek')
    if len(g_val) == 0:
        continue
    y_true = g_val['sales'].values
    pred = forecast_one(g_train['sales'], g_train['yearweek'], best_phase1, HORIZON)
    if len(pred) != len(y_true):
        pred = np.resize(pred, len(y_true))
    rows_p1.append({'framework': 'Phase1', 'type': typ, 'family': fam, 'model': best_phase1, 'wmape': wmape(y_true, pred)})

phase1 = pd.DataFrame(rows_p1)

final = pd.concat([phase1, phase2], ignore_index=True)
summary = final.groupby('framework')['wmape'].mean().sort_values()
by_type = final.groupby(['type','framework'])['wmape'].mean().unstack()

print('=== 프레임워크별 WMAPE (%) ===')
print(summary.round(2))
print('\n=== type × framework ===')
print(by_type.round(2))

final.to_parquet(DATA_PROCESSED / 'hybrid_forecast_results.parquet', index=False)
summary.to_csv(DATA_PROCESSED / 'hybrid_forecast_summary.csv')
by_type.to_csv(DATA_PROCESSED / 'hybrid_forecast_by_type.csv')
print('저장 완료')